In [ ]:
import yt
import itertools
import dispatch_utils as du
import dispatch_data as dd
import numpy as np
from scaling import *
import os
sc = scaling()

In [ ]:
run = 'input'
datadir='../data/'
prefix=''
snapshot = 0
all_p=du.patches(run=run,data=datadir, iout=snapshot)
pp=[p for p in all_p if p.iout==snapshot]
pp=[p for p in pp if p.kind[0:9]=='rt_solver']
len(pp)

In [ ]:
for p in pp:
        p.cache()
        p.stripit()
       # p.vars()
gamma = 1.4
def make_yt_dict(patches):
    gd=[]
    lbox=[0] * 3
    ubox=[0] * 3
    bbox=np.array([[0.,0.],[0.,0.],[0.,0.]])
    for p in patches:
        if p.kind[0:9] == 'rt_solver':
            for cc in range(3):
                if p.active_corners[0][cc] < lbox[cc]:
                    lbox[cc] = p.active_corners[0][cc]
                if p.active_corners[1][cc] > ubox[cc]:
                    ubox[cc] = p.active_corners[1][cc]
            gd.append(dict(
                left_edge = p.active_corners[0],
                right_edge = p.active_corners[1],
                level = p.level,
                dimensions=p.n,
                number_of_particles=0,
                e=(p.t,"erg/cm**3"),
                Q=(p.Q,"erg/cm**3/s**1"),
                #density = p.d,
                #s=p.s,
            ))#temperature=p.t))
    for cc in range(3):
        bbox[cc][0]=lbox[cc]
        bbox[cc][1]=ubox[cc]
    patches[0].bbox=bbox
    return gd

def load_yt(patches, refine_by=3,geometry="cartesian"):
    dic = make_yt_dict(patches) 
    ds=yt.load_amr_grids(dic,patches[0].n,
                       bbox=patches[0].bbox,
                       refine_by=refine_by,
                       geometry=geometry,
                       mass_unit=(sc.m,'g'),
                       length_unit=(sc.l,'cm'),
                       velocity_unit=(sc.v,'cm/s'),
                       time_unit=(sc.t,'s'),
                       sim_time = patches[0].time,
                       periodicity=(False,False,False))
    return ds


In [ ]:
ds = load_yt(pp, refine_by=3)
pp[0].bbox

In [ ]:
slc=yt.SlicePlot(ds=ds,axis="z",fields="Q",center=[0.5,0.5,0.5],width = ((1),(1)))
slc.set_antialias=False
slc.set_font_size(24.)
#slc.set_cmap("Q","bds_highcontrast")
#slc.set_log('Q', False)
#slc.set_zlim("Q", zmin=1e8, zmax=1e9)
slc.annotate_grids(alpha=0.9, draw_ids=True)

In [ ]:
slc.save("3dirs_4x64.pdf")

In [ ]:
pp[40].id